In [ ]:
import os
import json
import seaborn as sns
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm
from typing import List, Dict


sns.set_theme()

In [ ]:
PROCESSED_BUILDS_DIR = "/kaggle/input/datasets/ewendano/mcbuild-max128/builds_full_max128/builds_full_max128"
IDX_TO_BLOCK_JSON = (
    "/kaggle/input/datasets/ewendano/mcbuild-max128/idx_to_block_full_max128.json"
)
LOSSES_PLOT_FP = "/kaggle/working/losses.jpg"
MODEL_FP = "/kaggle/working/vae.pth"

In [ ]:
config = {
    "use_pretrained": True,
    "pretrained_fp": "/kaggle/working/vae.pth",
    "dataset": {"train_val_split": [0.8, 0.2], "batch_size": 2, "num_workers": 0},
    "model": {"latent_channels": 16, "embed_dim": 32},
    "train": {"epochs": 3, "lr": 0.001, "kl_weight": 0.0001},
}

## Utils

In [ ]:
def plot_losses(train_losses, val_losses, save_fp: str, show=True):
    """
    Plot the training and validation loss across epochs.

    - train_losses: List of training losses per epoch.
    - val_losses: List of validation losses per epoch.
    """
    plt.figure(figsize=(8, 5))
    plt.plot(train_losses, label="Training Loss", marker="o")
    plt.plot(val_losses, label="Validation Loss", marker="s")

    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.title("Training vs Validation Loss")
    plt.legend()
    plt.grid()
    plt.savefig(save_fp)
    if show:
        plt.show()

In [ ]:
def read_json(fp: str) -> Dict | List:
    with open(fp, "r") as f:
        d = json.load(f)
    return d

## Dataset

In [ ]:
class BuildDataset(Dataset):
    def __init__(self, filepaths) -> None:
        super().__init__()
        self.filepaths = filepaths

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, index):
        return torch.load(self.filepaths[index]).type(torch.long)


def custom_collate_fn(batch):
    """
    Custom collate function to prevent stacking of tensors with different shapes
    """
    return list(batch)


def get_dataset(train_val_split: List[float] = [0.8, 0.2]):
    filepaths = [
        os.path.join(PROCESSED_BUILDS_DIR, fn)
        for fn in os.listdir(PROCESSED_BUILDS_DIR)
        if fn.split(".")[-1] == "pt"
    ]
    dataset = BuildDataset(filepaths)

    train_dataset, val_dataset = random_split(dataset, lengths=train_val_split)
    return train_dataset, val_dataset


def get_loaders(train_val_split: List[float], batch_size: int, num_workers: int = 0):
    train_dataset, val_dataset = get_dataset(train_val_split)

    train_loader = DataLoader(
        dataset=train_dataset,
        batch_size=batch_size,
        collate_fn=custom_collate_fn,
        shuffle=True,
        num_workers=num_workers,
    )
    val_loader = DataLoader(
        dataset=val_dataset,
        batch_size=batch_size,
        collate_fn=custom_collate_fn,
        shuffle=True,
        num_workers=num_workers,
    )
    return train_loader, val_loader

## Model

In [ ]:
class VAE(nn.Module):
    def __init__(self, block_count, embed_dim=64, latent_channels=16) -> None:
        super().__init__()

        self.embedding = nn.Embedding(block_count, embed_dim)

        self.encoder = nn.Sequential(
            nn.Conv3d(embed_dim, 32, 3, 1, 1),
            nn.ReLU(),
            nn.Conv3d(32, 64, 3, 2, 1),
            nn.ReLU(),
            nn.Conv3d(64, 128, 3, 2, 1),
            nn.ReLU(),
        )

        self.conv_mu = nn.Conv3d(128, latent_channels, 1)
        self.conv_logvar = nn.Conv3d(128, latent_channels, 1)

        self.decoder = nn.Sequential(
            nn.ConvTranspose3d(latent_channels, 128, 3, 1, 1),
            nn.ReLU(),
            nn.ConvTranspose3d(128, 64, 3, 2, 1, 1),
            nn.ReLU(),
            nn.ConvTranspose3d(64, 32, 3, 2, 1, 1),
            nn.ReLU(),
            nn.ConvTranspose3d(32, block_count, 3, 1, 1),
        )

    def pad_to_multiple(self, x, multiple=4):
        d, h, w = x.shape[2:]
        pd = (multiple - d % multiple) % multiple
        ph = (multiple - h % multiple) % multiple
        pw = (multiple - w % multiple) % multiple
        # pad last 3 dims: (w_before, w_after, h_before, h_after, d_before, d_after)
        x = F.pad(x, (0, pw, 0, ph, 0, pd))
        return x, (d, h, w)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def encode(self, x):
        h = self.encoder(x)
        return self.conv_mu(h), self.conv_logvar(h)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x_list):
        results = []
        mu_list, logvar_list = [], []

        for x in x_list:
            x = self.embedding(x).permute(0, 4, 1, 2, 3)
            x, original_shape = self.pad_to_multiple(x)
            mu, logvar = self.encode(x)
            z = self.reparameterize(mu, logvar)
            out = self.decode(z)
            w, l, h = original_shape
            out = out[:, :, :w, :l, :h]
            results.append(out)
            mu_list.append(mu)
            logvar_list.append(logvar)

        return results, mu_list, logvar_list

## Training

In [ ]:
def vae_loss(outputs, targets, mu_list, logvar_list, kl_weight=1e-4):
    ce_loss = sum(
        F.cross_entropy(out, tgt) for out, tgt in zip(outputs, targets)
    ) / len(outputs)

    kl_loss = sum(
        -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
        for mu, logvar in zip(mu_list, logvar_list)
    ) / len(mu_list)

    return ce_loss + kl_weight * kl_loss, ce_loss, kl_loss

In [ ]:
def train(model: VAE, train_loader, val_loader, epochs, lr, kl_weight, device):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    use_amp = torch.cuda.is_available()
    scaler = torch.GradScaler(enabled=use_amp)

    train_losses = []
    val_losses = []
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        tqdm_loader = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs}")

        for x_list in tqdm_loader:
            optimizer.zero_grad(set_to_none=True)
            x_list = [x.to(device) for x in x_list]

            with torch.autocast("cuda", enabled=use_amp):
                recon_x_list, mu_list, logvar_list = model(x_list)
                loss, ce_loss, kl_loss = vae_loss(
                    recon_x_list, x_list, mu_list, logvar_list, kl_weight
                )

            scaler.scale(loss).backward()  # type: ignore
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()  # type: ignore
            tqdm_loader.set_postfix(
                {
                    "loss": loss.item(),  # type: ignore
                    "ce_loss": ce_loss.item(),  # type: ignore
                    "kl_loss": kl_loss.item(),  # type: ignore
                }
            )

        avg_train_loss = total_loss / len(train_loader.dataset)
        avg_val_loss = evaluate_model(model, val_loader, kl_weight, device, use_amp)

        print(
            f"[{epoch+1}/{epochs}]: train_loss= {avg_train_loss:.3f} | val_loss= {avg_val_loss:.3f}"
        )

        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)

        torch.save(model.state_dict(), MODEL_FP.replace("model", "vae"))

    return train_losses, val_losses


def evaluate_model(model, loader, kl_weight, device, use_amp):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for x_list in tqdm(loader, desc="Evaluating"):
            x_list = [x.to(device) for x in x_list]

            with torch.autocast("cuda", enabled=use_amp):
                recon_x_list, mu_list, logvar_list = model(x_list)
                loss, _, _ = vae_loss(
                    recon_x_list, x_list, mu_list, logvar_list, kl_weight
                )

            total_loss += loss.item()  # type: ignore

    avg_loss = total_loss / len(loader.dataset)
    return avg_loss

## Pipeline

In [ ]:
def pipeline_training(config):
    """
    Model training pipeline
    """
    # Dataloader
    train_loader, val_loader = get_loaders(**config["dataset"])

    # Model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    block_count = len(read_json(IDX_TO_BLOCK_JSON))
    vae = VAE(block_count, **config["model"]).to(device)

    if config["use_pretrained"]:
        vae.load_state_dict(torch.load(config["pretrained_fp"], map_location=device))

    print("\nTraining...")
    train_losses, val_losses = train(
        vae, train_loader, val_loader, **config["train"], device=device
    )

    # save model
    torch.save(vae.state_dict(), MODEL_FP)

    # save losses plot
    plot_losses(train_losses, val_losses, LOSSES_PLOT_FP)


if __name__ == "__main__":
    # run
    pipeline_training(config)